# CMP6202 – Notebook 01: Data preparation

**Project:** Multi-class Classification for Cloud Network Intrusion Detection on AWS Using CSE-CIC-IDS2018

This notebook produces the evidence for report sections **1.1**, **1.2** and **3.3**, and the cleaned, de-duplicated
75,000-row sample that every later notebook loads. Section **2.2** (splitting) is in notebook 02.

**Why the order differs from the report.** The template places *2.2 Splitting* before *3.3 Data cleaning*. I run
cleaning first because every rule in it is applied row by row and learns nothing from the data (no means, no scaling,
no imputation), so it cannot leak test-set information. Everything that *is* fitted to data (scaling, SMOTE,
hyper-parameter tuning) happens later inside a cross-validated pipeline on the training set only.

I keep the reusable logic in `src/nids/` (`config.py` for constants and their justification, `data.py` for the
preparation steps); this notebook calls it and explains the results.

In [1]:
# Google Colab setup. On a local machine this cell does nothing.
# On Colab: upload the project folder to Google Drive (README, "Run on Google Colab") and set PROJECT_DIR
# to its location. The cell mounts Drive, installs the pinned libraries and makes `nids` importable.
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    from importlib.metadata import version

    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/CMP6202_NIDS/Project"
    os.chdir(PROJECT_DIR)
    loaded = {name: version(dist) for name, dist in (("numpy", "numpy"), ("pandas", "pandas"), ("sklearn", "scikit-learn"))
              if name in sys.modules}
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
    sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))
    changed = [name for name, old in loaded.items()
               if old != version({"sklearn": "scikit-learn"}.get(name, name))]
    if changed:
        # A library that was already imported cannot be replaced in a running session.
        raise SystemExit(f"Upgraded {changed}: choose Runtime > Restart session, then run this cell again.")
    print("Running on Colab from", PROJECT_DIR)
else:
    print("Running locally; Colab setup skipped")

Running locally; Colab setup skipped


In [2]:
from time import perf_counter

import pandas as pd
from IPython.display import display

from nids import config, data

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 70)
for folder in (config.TABLES_DIR, config.TEXT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print(f"Raw file   : {config.RAW_CSV.name}  (original AWS name: {config.ORIGINAL_AWS_FILENAME})")
print(f"Size       : {config.RAW_CSV.stat().st_size / 1e6:,.1f} MB")
print(f"Random seed: {config.SEED}")

Raw file   : 02-14-2018.csv  (original AWS name: Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv)
Size       : 358.2 MB
Random seed: 42


# 1 Report Introduction

**Target audience.** I write for a *security engineering lead* who runs network monitoring on cloud infrastructure:
someone fluent in networking (ports, TCP, flows) who has to decide whether a machine-learning detector is worth
deploying, but who is not a machine-learning specialist. I therefore explain metrics and model behaviour in terms of
missed attacks, false alarms and running cost, and keep algorithmic detail to what that decision needs.

**Research questions.**

* **RQ1** – Once duplicate flows and label-proxy columns are removed, how accurately can Logistic Regression, a Decision
  Tree and a Random Forest separate FTP and SSH brute-force flows from benign traffic using flow statistics alone?
* **RQ2** – To what extent do the models rely on shortcut features (the destination port, and TCP values that are fixed
  by the attacking machine) rather than on the behaviour of the attack?
* **RQ3** – Does oversampling the rare FTP-BruteForce class with SMOTE improve detection compared with class weighting
  or no resampling?

In [3]:
intro = (
    "I write for a security engineering lead who runs network monitoring on cloud infrastructure: someone fluent in "
    "networking (ports, TCP, flows) who must decide whether a machine-learning detector is worth deploying, but who is "
    "not a machine-learning specialist, so I explain results in terms of missed attacks, false alarms and running cost. "
    "The project asks three questions. RQ1: once duplicate flows and label-proxy columns are removed, how accurately can "
    "Logistic Regression, a Decision Tree and a Random Forest separate FTP and SSH brute-force flows from benign traffic "
    "using flow statistics alone? RQ2: to what extent do the models rely on shortcut features (the destination port, and "
    "TCP values fixed by the attacking machine) rather than on the behaviour of the attack? RQ3: does oversampling the "
    "rare FTP-BruteForce class with SMOTE improve detection compared with class weighting or no resampling?"
)
(config.TEXT_DIR / "1_0_audience_and_research_questions.txt").write_text(intro + "\n")
print(intro, f"\n\n[{len(intro.split())} words]")

I write for a security engineering lead who runs network monitoring on cloud infrastructure: someone fluent in networking (ports, TCP, flows) who must decide whether a machine-learning detector is worth deploying, but who is not a machine-learning specialist, so I explain results in terms of missed attacks, false alarms and running cost. The project asks three questions. RQ1: once duplicate flows and label-proxy columns are removed, how accurately can Logistic Regression, a Decision Tree and a Random Forest separate FTP and SSH brute-force flows from benign traffic using flow statistics alone? RQ2: to what extent do the models rely on shortcut features (the destination port, and TCP values fixed by the attacking machine) rather than on the behaviour of the attack? RQ3: does oversampling the rare FTP-BruteForce class with SMOTE improve detection compared with class weighting or no resampling? 

[139 words]


## 1.1 Dataset identification

**What it represents.** CSE-CIC-IDS2018 is an intrusion-detection dataset produced by the Communications Security
Establishment (CSE) and the Canadian Institute for Cybersecurity (CIC, University of New Brunswick). CIC describes
the testbed as “a common LAN network topology on the AWS computing platform” with an attacking infrastructure of
50 machines and a victim organisation of 5 departments, 420 machines and 30 servers. Each row is one network
**flow** summarised by CICFlowMeter-V3 into roughly 80 statistical features.

**This file.** Wednesday 14 February 2018. CIC Table 2 lists two attacks that day, both run from Kali Linux against
an Ubuntu server: **FTP-BruteForce** (10:32–12:09) and **SSH-Bruteforce** (14:01–15:31). Labels were assigned by CIC
from “the attack scenarios schedule and the IPs and ports of the source and destination along with the protocol name”,
which matters later: destination port and time are part of how the ground truth was created.

**Source and licence.** The dataset is distributed through the Registry of Open Data on AWS (`s3://cse-cic-ids2018/`);
I use the Kaggle mirror *IDS 2018 Intrusion CSVs (CSE-CIC-IDS2018)* (file `02-14-2018.csv`), published under
**CC BY 4.0 (Attribution 4.0 International)**, which permits academic use with attribution. CIC additionally requires
that *“any use or redistribution of the data must include a citation to the CSE-CIC-IDS2018 dataset and a link to this
page in AWS”*, so I cite it as (Sharafaldin, Lashkari and Ghorbani, 2018) and link https://registry.opendata.aws/cse-cic-ids2018.
Because CC BY 4.0 requires changes to be indicated, the processed sample is documented as derived data (section 3.3).

The attribute table below lists every column with its CIC description (verbatim from CIC Table 3) and my notes where
the source table is incomplete or contains copy errors.

In [4]:
columns = data.raw_columns()
print(f"{len(columns)} columns: {len(columns) - 4} numeric flow statistics, the categorical 'Dst Port' and 'Protocol', "
      f"'Timestamp', and the target 'Label'.")

80 columns: 76 numeric flow statistics, the categorical 'Dst Port' and 'Protocol', 'Timestamp', and the target 'Label'.


## 1.2 Supervised learning task identification

The task is **multi-class classification**: given the statistical features of a network flow, predict whether it is
`Benign`, `FTP-BruteForce` or `SSH-Bruteforce`. The target variable (ground truth) is the `Label` column.

In a cloud deployment, flow records of this kind are what a network intrusion detection system (NIDS) can observe
without inspecting payloads. What I want is a model that separates brute-force login attempts from normal traffic
using flow statistics alone, rather than a fixed port number; this is why I exclude `Dst Port` from the main model
(section 3.3).

# 3 Experimental Design

## 3.3 Data cleaning and pre-processing transformations

I read the raw file in chunks of 200,000 rows with explicit `float64` types, so peak memory stays small and every
chunk is parsed identically. I justify each step below by what I measured in this specific file.

### Step 1 – Load, clean row by row, and de-duplicate

In [5]:
start = perf_counter()
flows, report = data.build_unique_flows()
print(f"Read, cleaned and de-duplicated {report.raw.sum():,} rows in {perf_counter() - start:.1f} s")

Read, cleaned and de-duplicated 1,048,575 rows in 4.4 s


### Step 2 – Non-finite and impossible values

CICFlowMeter computes `Flow Byts/s` and `Flow Pkts/s` by dividing by the flow duration; when the duration is zero the
result is written as `Infinity` or `NaN`. I convert these values to `NaN` and drop the rows rather than impute them,
because a rate for a zero-length flow does not exist and the affected rows are a very small share of the file.
Negative durations and inter-arrival times are physically impossible (clock errors), so I drop them too. The value
`-1` in `Init Fwd/Bwd Win Byts` is **not** an error and I keep it: in this file it appears in every non-TCP flow
(protocols 0 and 17) and, for TCP, only in the backward column of flows with no backward packets, i.e. it means
“no TCP window observed”. (This meaning is inferred from the data; CIC's feature table does not document it.)

In [6]:
problems = pd.DataFrame({
    "Infinity": report.inf_by_column,
    "NaN": report.nan_by_column,
    "Negative": report.negative_by_column,
})
problems = problems.fillna(0).astype(int)
problems = problems[(problems > 0).any(axis=1)]
display(problems)
problems.to_csv(config.TABLES_DIR / "3_3_invalid_values_by_column.csv")

,Infinity,NaN,Negative
Flow Byts/s,1547,2277,0
Flow Duration,0,0,5
Flow IAT Max,0,0,2
Flow IAT Mean,0,0,5
Flow IAT Min,0,0,5
Flow Pkts/s,3824,0,5
Fwd IAT Max,0,0,2
Fwd IAT Mean,0,0,5
Fwd IAT Min,0,0,5
Fwd IAT Tot,0,0,5


### Step 3 – Duplicate flows and contradictory labels

The raw file is dominated by repeated flows. I treat two rows as the same flow when all features **except
`Timestamp` and `Dst Port`** are identical and the label matches, and keep the first occurrence. I ignore `Timestamp`
because it records *when* a flow was seen, not *what* it was, and `Dst Port` because my main model will not use it;
de-duplicating in this smaller feature space also guarantees uniqueness in any larger one, so the later
“with `Dst Port`” experiment remains leakage-free.

Without this step, a random split would place copies of the same flow in both the training and the test set, and the
test score would measure memory rather than generalisation.

I remove feature vectors that occur under **more than one label** entirely: their ground truth is contradictory, so
no model can be right on both copies.

In [7]:
stages = report.stage_table()
display(stages)
print(f"Feature vectors with contradictory labels: {report.n_conflicting_vectors} "
      f"({int(report.conflicts_removed.sum())} rows: {report.conflicts_removed.to_dict()})")

Label,Benign,FTP-BruteForce,SSH-Bruteforce,Total
1. Raw file,667626,193360,187589,1048575
2. Removed: non-finite values (Infinity/NaN),3818,6,0,3824
3. Removed: impossible negative values,5,0,0,5
4. After row-wise cleaning,663803,193354,187589,1044746
5. Removed: duplicate flows (same features and label),138563,193301,93541,425405
6. Removed: same features with different labels,0,7,7,14
7. Unique flows (sampling frame),525240,46,94041,619327


Feature vectors with contradictory labels: 7 (14 rows: {'FTP-BruteForce': 7, 'SSH-Bruteforce': 7})


The contradictory vectors are all FTP/SSH pairs that are identical apart from the destination port: once the port is
removed, these short attack flows cannot be told apart. FTP-BruteForce collapses from 193,360 rows to a few dozen unique
flows, which means almost all of its rows are exact repeats of the same handful of connection attempts.

### Step 4 – Stratified sample of 75,000 flows

The brief recommends at least 3,000 rows; I use 75,000 to keep enough minority-class examples for 5-fold
cross-validation while training stays fast. I stratify by `Label` (proportional to class size) with one exception: a
class receives at least `min(class size, 2,000)` rows. After de-duplication FTP-BruteForce is far below that floor, so I
keep all of its unique flows; proportional sampling would have kept only about six.

In [8]:
sample = data.stratified_sample(flows)

distribution = pd.DataFrame({
    "Raw rows": report.raw,
    "Raw %": report.raw / report.raw.sum() * 100,
    "Unique flows": report.unique,
    "Unique %": report.unique / report.unique.sum() * 100,
    "Sample rows": sample[config.LABEL_COL].value_counts(),
    "Sample %": sample[config.LABEL_COL].value_counts(normalize=True) * 100,
}).sort_index()
distribution.loc["Total"] = distribution.sum()
display(distribution.round(3))
distribution.round(3).to_csv(config.TABLES_DIR / "3_3_label_distribution.csv")
print(f"Seed {config.SEED}; sample shape {sample.shape}")

,Raw rows,Raw %,Unique flows,Unique %,Sample rows,Sample %
Label,,,,,,
Benign,667626,63.67,525240,84.808,63572,84.763
FTP-BruteForce,193360,18.44,46,0.007,46,0.061
SSH-Bruteforce,187589,17.89,94041,15.184,11382,15.176
Total,1048575,100.00,619327,100.000,75000,100.000


Seed 42; sample shape (75000, 80)


### Attribute table (report section 1.1, appendix)

Observed values come from the de-duplicated sampling frame (619,327 flows). Columns flagged `constant` hold a single
value everywhere and carry no information; the modelling pipeline removes them (`VarianceThreshold`, notebook 02).

Each column is also classified by its **level of measurement** (nominal, ordinal, interval, ratio), because that
decides how it may be encoded and summarised: `Protocol` and `Dst Port` are numbers that only *name* a category
(nominal), so they are one-hot encoded rather than treated as quantities; `Timestamp` is interval; every flow
statistic is a count, size, duration or rate with a true zero (ratio).

In [9]:
attributes = data.attribute_table(flows)
attributes.to_csv(config.TABLES_DIR / "1_1_attributes.csv", index=False)
display(attributes[["csv_column", "cic_description", "role", "level_of_measurement", "dtype", "n_unique", "constant"]])
print("Constant columns:", attributes.loc[attributes["constant"], "csv_column"].tolist())
levels = attributes["level_of_measurement"].value_counts()
display(levels.rename("columns").to_frame())
levels.to_csv(config.TABLES_DIR / "1_1_levels_of_measurement.csv")

,csv_column,cic_description,role,level_of_measurement,dtype,n_unique,constant
0,Dst Port,,categorical,nominal,float64,7770,False
1,Protocol,,categorical,nominal,float64,3,False
2,Timestamp,,identifier,interval,string,31919,False
3,Flow Duration,Flow duration,numeric,ratio (discrete),float64,389487,False
4,Tot Fwd Pkts,Total packets in the forward direction,numeric,ratio (discrete),float64,750,False
...,...,...,...,...,...,...,...
75,Idle Mean,Mean time a flow was idle before becoming active,numeric,ratio (continuous),float64,80416,False
76,Idle Std,Standard deviation time a flow was idle before becoming active,numeric,ratio (continuous),float64,57844,False
77,Idle Max,Maximum time a flow was idle before becoming active,numeric,ratio (discrete),float64,59131,False
78,Idle Min,Minimum time a flow was idle before becoming active,numeric,ratio (discrete),float64,69043,False


Constant columns: ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg', 'Bwd Byts/b Avg', 'Bwd Pkts/b Avg', 'Bwd Blk Rate Avg']


,columns
level_of_measurement,
ratio (discrete),52
ratio (continuous),24
nominal,3
interval,1


### Save the sample and its manifest

I record the SHA-256 of the raw file and of the sample, the seed, every stage count and the library versions in a
manifest, so anyone can verify or rebuild the sample exactly.

In [10]:
manifest = data.save_sample(sample, report)
final_stages = report.stage_table(sample[config.LABEL_COL].value_counts())
final_stages.to_csv(config.TABLES_DIR / "3_3_cleaning_stages.csv")
display(final_stages)
print(f"Saved {config.SAMPLE_PARQUET.relative_to(config.PROJECT_ROOT)}  sha256={manifest['sample_sha256'][:16]}…")

Label,Benign,FTP-BruteForce,SSH-Bruteforce,Total
1. Raw file,667626,193360,187589,1048575
2. Removed: non-finite values (Infinity/NaN),3818,6,0,3824
3. Removed: impossible negative values,5,0,0,5
4. After row-wise cleaning,663803,193354,187589,1044746
5. Removed: duplicate flows (same features and label),138563,193301,93541,425405
6. Removed: same features with different labels,0,7,7,14
7. Unique flows (sampling frame),525240,46,94041,619327
8. Stratified sample,63572,46,11382,75000


Saved data/processed/sample_75k_dedup.parquet  sha256=34cc46c3488eab82…


### Copy-ready summary for report section 3.3

I generate this paragraph from the numbers above (saved to `reports/text/3_3_data_cleaning.txt`), so the report never
contains a number that the code did not produce.

In [11]:
raw = report.raw.sort_index()
summary = (
    f"The Wednesday 14-02-2018 file contains {raw.sum():,} flows "
    f"({', '.join(f'{label} {n:,}' for label, n in raw.items())}). "
    f"{int(report.nonfinite_removed.sum()):,} rows with Infinity or NaN in Flow Byts/s or Flow Pkts/s "
    f"(zero-duration flows) and {int(report.negative_removed.sum())} rows with negative durations were removed. "
    f"De-duplication on all features except Timestamp and Dst Port removed "
    f"{int(report.duplicates_removed.sum()):,} repeated flows ({report.duplicates_removed.sum() / report.after_cleaning.sum():.1%} "
    f"of the cleaned rows); FTP-BruteForce fell from {raw['FTP-BruteForce']:,} rows to "
    f"{report.unique['FTP-BruteForce']} unique flows. {report.n_conflicting_vectors} feature vectors labelled both "
    f"FTP-BruteForce and SSH-Bruteforce ({int(report.conflicts_removed.sum())} rows) were removed as contradictory, "
    f"leaving {report.unique.sum():,} unique flows. A stratified sample of {config.SAMPLE_SIZE:,} flows "
    f"(seed {config.SEED}) was drawn, keeping all {report.unique['FTP-BruteForce']} FTP-BruteForce flows: "
    f"{', '.join(f'{label} {n:,}' for label, n in sample[config.LABEL_COL].value_counts().sort_index().items())}."
)
(config.TEXT_DIR / "3_3_data_cleaning.txt").write_text(summary + "\n")
print(summary)
print(f"\n[{len(summary.split())} words]")

The Wednesday 14-02-2018 file contains 1,048,575 flows (Benign 667,626, FTP-BruteForce 193,360, SSH-Bruteforce 187,589). 3,824 rows with Infinity or NaN in Flow Byts/s or Flow Pkts/s (zero-duration flows) and 5 rows with negative durations were removed. De-duplication on all features except Timestamp and Dst Port removed 425,405 repeated flows (40.7% of the cleaned rows); FTP-BruteForce fell from 193,360 rows to 46 unique flows. 7 feature vectors labelled both FTP-BruteForce and SSH-Bruteforce (14 rows) were removed as contradictory, leaving 619,327 unique flows. A stratified sample of 75,000 flows (seed 42) was drawn, keeping all 46 FTP-BruteForce flows: Benign 63,572, FTP-BruteForce 46, SSH-Bruteforce 11,382.

[101 words]
